In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
def missing_values(df):
    for feature in df.columns:
        missing_values = df[feature].isna().sum()
        percentage = missing_values/len(df) * 100
        print(f'{feature} - {missing_values} - {round(percentage, 1)} %')
        
def fill_with_mode(df, feature_list):
    for feature in feature_list:
        mode = df[feature].mode()[0]
        df[feature].fillna(value=mode, inplace=True)

def fill_with_mean(df, feature_list):
    for feature in feature_list:
        df[feature] = pd.to_numeric(df[feature], errors='coerce')
        mean = df[feature].mean()
        df[feature].fillna(value=mean, inplace=True)
        
def y_data(df):
    df = df.loc[6:22]
    df_1 = df.copy()
#     df_1['LFG'] = df_1['LF'].diff()
#     df_1['EPFRG'] = df_1['EPFR'].diff()
    df_1['GGAG'] = df_1['GGA'].diff()
    df_1['PAG'] = df_1['PA'].diff()
    df_1 = df_1.drop(6)
#     df_1 = df_1.drop(["Unnamed: 0", "REF_AREA_LABEL", "HI", "OCTB", "NINA", "LF", "EPFR", "GGA", "EIA", "GDP",
#                      "UR", "URFF", "YUA", "P014", "P65"], axis=1)
    df_1 = df_1[["TIME_PERIOD", "GDPC", "GDPCG", "VABI", "EMP", "PAG", "QOA", "QOR", "YUA", "GGAG", "REST", "CO2", "EBSG", "FDI", "GGFC", "LBGD", "PC", "ARP", "PS"]]
    df_1["EMP"] = df_1["EMP"].replace(':', np.nan)
#     df_1["LTU"] = df_1["LTU"].replace(':', np.nan)
#     df_1["FEM"] = df_1["FEM"].replace(':', np.nan)
    return df_1

def merge_df(df_1, df_2):
    frames = [df_1, df_2]
    result = pd.concat(frames)
    return result

def needed_context(country_context, needed_context):
    context = country_context[needed_context]
    return context.loc[7:22]

def eu_payments_for_country(eu_payments, country):
    eu_payments = eu_payments.drop(['NUTS1_ID', 'NUTS2 code+name', 'NUTS2_ID', 'NUTS2_name', 'Modelled_annual_expenditure',
                  'Standard_Deviation_of_annual_expenditure', 'Standard_Error_of_modelled_annual_expenditure'], axis=1)
    country_eu_payments = eu_payments[eu_payments['Country'] == country].copy()
    country_eu_payments['EU_Payment_annual'] = pd.to_numeric(country_eu_payments['EU_Payment_annual'].str.replace(',', ''))
    country_eu_payments_all = country_eu_payments.groupby('Year')['EU_Payment_annual'].sum().reset_index()
    country_eu_payments_all = country_eu_payments_all.rename({"Year":"TIME_PERIOD"}, axis=1)
    return country_eu_payments_all

def merge_data(df_1, df_2, df_3):
    df = pd.merge(df_1, df_2, on='TIME_PERIOD', how='outer')
    df = pd.merge(df, df_3, on='TIME_PERIOD', how='outer')
    return df

def fix_eu_payments(df):
    df['EUPC'] = df['EU_Payment_annual'] / (df['PS'] * 1000000)
    df = df.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)
    return df

def standardize_data(df):
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(df.values)
    df_scaled = pd.DataFrame(scaled_features, index=df.index, columns=df.columns)
    return df_scaled

In [3]:
df_dk = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/Denmark.csv")
df_de = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/Germany.csv")
df_pl = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/Poland.csv")
df_lt = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/Lithuania.csv")
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/Latvia.csv")
df_ee = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/Estonia.csv")
df_fi = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/Finland.csv")
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/Sweden.csv")
dk_context = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/Context/Denmark.csv")
de_context = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/Context/Germany.csv")
pl_context = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/Context/Poland.csv")
lt_context = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/Context/Lithuania.csv")
lv_context = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/Context/Latvia.csv")
ee_context = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/Context/Estonia.csv")
fi_context = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/Context/Finland.csv")
se_context = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/Context/Sweden.csv")
eu_payments = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Raw_Data/Variables/Historic_EU_payments_annual_timeseries_-_regionalised_and_modelled_20251209.csv")

In [4]:
df_dk = y_data(df_dk)

In [5]:
missing_values(df_dk)

TIME_PERIOD - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
VABI - 0 - 0.0 %
EMP - 2 - 12.5 %
PAG - 1 - 6.2 %
QOA - 3 - 18.8 %
QOR - 3 - 18.8 %
YUA - 0 - 0.0 %
GGAG - 2 - 12.5 %
REST - 0 - 0.0 %
CO2 - 0 - 0.0 %
EBSG - 0 - 0.0 %
FDI - 0 - 0.0 %
GGFC - 0 - 0.0 %
LBGD - 0 - 0.0 %
PC - 0 - 0.0 %
ARP - 0 - 0.0 %
PS - 0 - 0.0 %


In [6]:
fill_with_mode(df_dk, ["QOR", "QOA"])

In [7]:
fill_with_mean(df_dk, ["EMP", "PAG", "GGAG"])

In [8]:
# context_DFSG = ["TIME_PERIOD", "DI", "CFI", "COC", "CPI", "VAA"]
# context_LLE = ["TIME_PERIOD", "EC", "DI", "CFI", "COC", "CPI"]
# context_Poland = ["TIME_PERIOD", "JI", "FOP", "DBD", "PR", "PT"]
context = ['TIME_PERIOD', 'ROL', 'GE', 'COC', 'RQ', 'SSB', 'PT']

In [9]:
dk_context = needed_context(dk_context, context)

In [10]:
dk_context

,TIME_PERIOD,ROL,GE,COC,RQ,SSB,PT
7,2007,2.00,2.35,2.43,1.93,89.822480,89.509231
8,2008,1.95,2.24,2.38,1.87,90.069128,89.265890
9,2009,1.92,2.23,2.44,1.88,90.102783,90.558650
10,2010,1.90,2.10,2.35,1.87,90.196051,90.980440
11,2011,1.92,2.10,2.39,1.90,91.112127,91.037842
12,2012,1.88,1.99,2.38,1.80,91.170983,91.680405
13,2013,1.90,1.94,2.40,1.80,91.198876,91.832348
14,2014,2.10,1.78,2.25,1.68,91.219461,91.834965
15,2015,2.02,1.79,2.19,1.72,91.802316,91.834965
16,2016,1.88,1.82,2.20,1.56,91.819264,91.142197


In [11]:
missing_values(dk_context)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
SSB - 2 - 12.5 %
PT - 2 - 12.5 %


In [12]:
# fill_with_mode(dk_context, ["DI"])

In [13]:
fill_with_mean(dk_context, ["SSB", "PT"])

In [14]:
dk_payments = eu_payments_for_country(eu_payments, "DK")

In [15]:
dk = merge_data(df_dk, dk_context, dk_payments)

In [16]:
dk = fix_eu_payments(dk)

In [17]:
dk.columns

Index(['GDPC', 'GDPCG', 'VABI', 'EMP', 'PAG', 'QOA', 'QOR', 'YUA', 'GGAG',
       'REST', 'CO2', 'EBSG', 'FDI', 'GGFC', 'LBGD', 'PC', 'ARP', 'ROL', 'GE',
       'COC', 'RQ', 'SSB', 'PT', 'EUPC'],
      dtype='object')

In [18]:
dk.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/dk_bn.csv')

In [19]:
dk_scaled = standardize_data(dk)

In [20]:
dk_scaled.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/dk_tcdf.csv')

In [21]:
column_index = dk_scaled.columns.get_loc('GGAG')
print(column_index)

8


In [22]:
8,1,0
10,2,2
6,4,2
4,4,2
17,8,3
8,8,
1,12,1
1,14,0
16,17,2
9,22,0

(9, 22, 0)

In [23]:
df_de = y_data(df_de)

In [24]:
missing_values(df_de)

TIME_PERIOD - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
VABI - 0 - 0.0 %
EMP - 2 - 12.5 %
PAG - 1 - 6.2 %
QOA - 3 - 18.8 %
QOR - 3 - 18.8 %
YUA - 0 - 0.0 %
GGAG - 2 - 12.5 %
REST - 0 - 0.0 %
CO2 - 0 - 0.0 %
EBSG - 0 - 0.0 %
FDI - 0 - 0.0 %
GGFC - 0 - 0.0 %
LBGD - 0 - 0.0 %
PC - 0 - 0.0 %
ARP - 0 - 0.0 %
PS - 0 - 0.0 %


In [25]:
fill_with_mean(df_de, ["EMP", "PAG", "GGAG", "QOR", "QOA"])

In [26]:
de_context = needed_context(de_context, context)

In [27]:
de_context

,TIME_PERIOD,ROL,GE,COC,RQ,SSB,PT
7,2007,1.76,1.63,1.73,1.62,78.044700,77.197929
8,2008,1.73,1.51,1.75,1.49,79.197611,76.399686
9,2009,1.65,1.58,1.75,1.52,79.363510,76.198556
10,2010,1.64,1.52,1.77,1.57,79.567065,78.957718
11,2011,1.62,1.50,1.74,1.55,80.744231,76.428017
12,2012,1.67,1.53,1.82,1.53,80.867728,76.817402
13,2013,1.64,1.51,1.80,1.54,80.918824,79.063708
14,2014,1.85,1.66,1.81,1.70,80.997172,76.843778
15,2015,1.76,1.66,1.78,1.71,80.627202,77.020556
16,2016,1.58,1.66,1.78,1.81,81.991794,82.167297


In [28]:
fill_with_mean(de_context, ["SSB", "PT"])

In [29]:
de_payments = eu_payments_for_country(eu_payments, "DE")

In [30]:
de = merge_data(df_de, de_context, de_payments)

In [31]:
de = fix_eu_payments(de)

In [32]:
de.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/de_bn.csv')

In [33]:
de_scaled = standardize_data(de)

In [34]:
de_scaled.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/de_tcdf.csv')

In [35]:
de_scaled

,GDPC,GDPCG,VABI,EMP,PAG,QOA,QOR,YUA,GGAG,REST,...,LBGD,PC,ARP,ROL,GE,COC,RQ,SSB,PT,EUPC
0,-1.344368,0.729169,1.012551,7.146179e-15,0.282342,1.487576e+00,1.508163e+00,1.935508,-0.429255,-1.457643,...,0.315491,-0.861067,-1.076826,1.325725,0.931256,-1.467563,-0.061775,-2.257396,-0.761426,-1.885554
1,-1.214641,0.009729,0.357311,7.146179e-15,1.410249,1.378010e+00,1.481668e+00,1.463077,0.763641,-1.447322,...,0.239042,-1.263737,-1.076826,0.990805,-0.062084,-1.020306,-1.346706,-1.319952,-1.101007,-1.575685
2,-1.434157,-2.384760,-3.230130,-2.043798e+00,-0.609187,1.213660e+00,1.349195e+00,1.664789,-1.498855,-1.203674,...,-0.832506,-1.206212,-0.559950,0.097685,0.517364,-1.020306,-1.050183,-1.185058,-1.186570,0.696700
3,-1.122751,1.206969,0.357311,-1.641504e+00,-0.194064,1.213660e+00,1.163731e+00,1.192358,1.816990,-0.948454,...,-1.513856,-0.688494,-0.387657,-0.013955,0.020695,-0.573048,-0.555979,-1.019545,-0.012793,0.436416
4,-0.666485,1.727436,0.766836,-9.374890e-01,0.353839,7.753937e-01,4.483729e-01,0.401434,-0.476978,-0.697300,...,0.193687,-0.142013,-0.043073,-0.237235,-0.144862,-1.243934,-0.753661,-0.062380,-1.088955,0.847532
5,-0.546408,-0.295415,1.209123,-7.363420e-01,0.131322,5.014775e-01,2.894043e-01,-0.028532,1.159200,-0.359822,...,0.473552,0.174370,0.473804,0.320965,0.103473,0.545095,-0.951343,0.038037,-0.923306,0.933917
6,-0.396749,-0.360138,0.111596,-4.849081e-01,0.933114,-2.380964e-01,7.744622e-02,-0.187778,1.280168,-0.294766,...,0.540724,0.346943,0.473804,-0.013955,-0.062084,0.097838,-0.852502,0.079584,0.032296,1.890205
7,-0.130328,0.249629,0.275406,-2.837610e-01,0.982724,-6.215792e-01,-2.669857e-01,-0.299251,-0.901683,-0.099598,...,0.830367,1.008473,1.507557,2.330485,1.179591,0.321466,0.728951,0.143290,-0.912085,0.801911
8,0.055523,-0.112070,0.127977,-1.329007e-01,-0.163423,-5.941876e-01,-6.909018e-01,-0.447881,0.833872,0.063043,...,0.827503,2.935536,1.507557,1.325725,1.179591,-0.349420,0.827792,-0.157537,-0.836882,-0.116677
9,0.266910,0.147926,0.619407,1.688199e-01,1.197946,-7.311457e-01,-1.008839e+00,-0.580586,0.770240,0.058038,...,0.862113,0.692089,1.162972,-0.683795,1.179591,-0.349420,1.816199,0.952027,1.352595,-1.658277


In [36]:
de.columns

Index(['GDPC', 'GDPCG', 'VABI', 'EMP', 'PAG', 'QOA', 'QOR', 'YUA', 'GGAG',
       'REST', 'CO2', 'EBSG', 'FDI', 'GGFC', 'LBGD', 'PC', 'ARP', 'ROL', 'GE',
       'COC', 'RQ', 'SSB', 'PT', 'EUPC'],
      dtype='object')

In [124]:
column_index = de_scaled.columns.get_loc('EUPC')
print(column_index)

23


In [ ]:
2,1,0
4,1,1
21,2,2 
18,2,1
12,4,0 
7,6,0
23,7,0
18,8,2
4,8,0
17,12,3 
4,12,3
20,15,2
11,16,3
12,16,3
20,16,3
12,17,3
8,18,1
23,19,2
18,21,0
17,22,0
21,23,1
5,23,0
20,23,0

In [37]:
df_pl = y_data(df_pl)

In [38]:
missing_values(df_pl)

TIME_PERIOD - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
VABI - 0 - 0.0 %
EMP - 2 - 12.5 %
PAG - 1 - 6.2 %
QOA - 3 - 18.8 %
QOR - 3 - 18.8 %
YUA - 0 - 0.0 %
GGAG - 2 - 12.5 %
REST - 0 - 0.0 %
CO2 - 0 - 0.0 %
EBSG - 0 - 0.0 %
FDI - 0 - 0.0 %
GGFC - 0 - 0.0 %
LBGD - 0 - 0.0 %
PC - 0 - 0.0 %
ARP - 0 - 0.0 %
PS - 0 - 0.0 %


In [39]:
# fill_with_mode(df_pl, ["PDPS", "FEM"])

In [40]:
fill_with_mean(df_pl, ["EMP", "PAG", "GGAG", "QOR", "QOA"])

In [41]:
pl_context = needed_context(pl_context, context)

In [42]:
missing_values(pl_context)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
SSB - 2 - 12.5 %
PT - 2 - 12.5 %


In [43]:
# fill_with_mode(pl_context, ["DBD"])

In [44]:
fill_with_mean(pl_context, ["SSB", "PT"])

In [45]:
pl_payments = eu_payments_for_country(eu_payments, "PL")

In [46]:
pl = merge_data(df_pl, pl_context, pl_payments)

In [47]:
pl = fix_eu_payments(pl)

In [48]:
pl.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/pl_bn.csv')

In [49]:
pl_scaled = standardize_data(pl)

In [50]:
pl_scaled 

,GDPC,GDPCG,VABI,EMP,PAG,QOA,QOR,YUA,GGAG,REST,...,LBGD,PC,ARP,ROL,GE,COC,RQ,SSB,PT,EUPC
0,-1.323454,1.195608,-0.232693,0.000000,0.388178,-1.118578e+00,-0.728098,0.520081,3.396365e-01,-1.680168,...,0.729229,0.305565,0.748491,-1.062459,-1.085532,-2.552775,-1.429553,-2.122321,-1.554977,-1.830687
1,-1.135254,0.119957,-0.819207,0.000000,0.037050,-9.564048e-01,-1.593346,-0.204906,-3.227485e-01,-1.415219,...,-0.075492,0.444261,0.437430,-0.274410,-0.449323,-1.109210,-0.736436,-2.061568,-1.565233,-1.621839
2,-0.977915,-0.681177,0.634328,-1.157587,0.832773,-8.212606e-01,-1.396699,0.364606,-1.165408e+00,-1.080226,...,-1.531817,0.496272,0.592960,0.288482,-0.067597,-1.194126,0.649797,-1.830870,-1.650553,-0.841057
3,-0.847102,-0.277784,0.302820,-1.243562,0.562480,-1.064520e+00,-1.173832,0.890180,2.100587e+00,-0.875508,...,-1.524087,0.530946,0.981786,0.626217,0.632233,-0.599716,1.045863,0.213607,-1.390884,-0.388894
4,-0.619924,0.484237,1.182591,-1.179081,1.502191,-1.064520e+00,-1.042734,1.219720,-9.851125e-02,-0.518182,...,-0.452705,0.357576,1.059552,1.076531,0.441370,-0.175138,0.451763,0.222691,-0.350874,0.287872
5,-0.510904,-1.135169,0.978586,-1.093106,1.135906,-6.320588e-01,-0.662549,1.351535,-3.994612e-01,-0.309065,...,-0.096768,0.357576,0.592960,1.245398,0.823096,0.674018,0.847830,0.232469,-0.176970,0.468269
6,-0.480027,-1.472061,-0.997711,-1.028625,-0.642473,-3.617705e-01,-0.098828,1.495181,-2.825235e-01,-0.140892,...,-0.405974,0.149532,0.748491,1.357977,0.823096,0.589102,1.342913,0.282374,0.463032,0.569487
7,-0.342156,-0.043096,0.608827,-0.663233,-0.953184,-6.445337e-02,0.556663,0.915529,-1.008442e+00,-0.089120,...,-0.059810,0.305565,0.515195,1.526844,1.522926,0.843849,1.441930,0.699236,0.474145,0.892184
8,-0.153844,0.177870,1.947609,-0.362322,1.651232,7.069079e-02,0.897518,0.400095,5.975094e-01,0.004272,...,0.381008,0.184206,0.981786,1.132820,1.332063,0.928764,0.352747,0.703514,0.470821,-0.490514
9,-0.046688,-0.448443,1.539599,0.046058,-1.253791,6.923539e-01,1.146604,-0.122099,1.349098e+00,-0.159841,...,0.318610,0.392250,0.748491,-0.049253,0.759475,1.523173,-0.043320,0.721521,1.147557,-1.433209


In [51]:
pl_scaled.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/pl_tcdf.csv')

In [125]:
pl_scaled.columns

Index(['GDPC', 'GDPCG', 'VABI', 'EMP', 'PAG', 'QOA', 'QOR', 'YUA', 'GGAG',
       'REST', 'CO2', 'EBSG', 'FDI', 'GGFC', 'LBGD', 'PC', 'ARP', 'ROL', 'GE',
       'COC', 'RQ', 'SSB', 'PT', 'EUPC'],
      dtype='object')

In [ ]:
column_index = de_scaled.columns.get_loc('EUPC')
print(column_index

In [ ]:
8,2,3
23,4,2
12,4,1
23,8,0
10,8,1
2,8,1
16,9,0
19,12,2
2,12,0
22,12,1
8,13,1
13,14,0
12,17,2
0,19,0
8,20,0
23,20,0 
9,21,3
16,23,1

In [52]:
df_lt = y_data(df_lt)

In [53]:
missing_values(df_lt)

TIME_PERIOD - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
VABI - 0 - 0.0 %
EMP - 2 - 12.5 %
PAG - 1 - 6.2 %
QOA - 3 - 18.8 %
QOR - 3 - 18.8 %
YUA - 0 - 0.0 %
GGAG - 2 - 12.5 %
REST - 0 - 0.0 %
CO2 - 0 - 0.0 %
EBSG - 0 - 0.0 %
FDI - 0 - 0.0 %
GGFC - 0 - 0.0 %
LBGD - 0 - 0.0 %
PC - 0 - 0.0 %
ARP - 0 - 0.0 %
PS - 0 - 0.0 %


In [54]:
df_lt

,TIME_PERIOD,GDPC,GDPCG,VABI,EMP,PAG,QOA,QOR,YUA,GGAG,REST,CO2,EBSG,FDI,GGFC,LBGD,PC,ARP,PS
7,2007,8985.274630,12.405345,29.65,NaN,-3.0,4.40,4.97,8.39,1013.0,16.482,15.5684,-13.094653,6.55,17.451336,-0.493558,-11.6,19.1,3.23
8,2008,10219.116755,3.660125,29.20,NaN,25.0,4.61,5.21,13.35,-563.0,17.824,15.8313,-11.598256,3.61,18.666542,-2.844590,-9.0,20.9,3.20
9,2009,8528.034573,-13.887757,25.02,67.0,4.0,4.22,5.28,29.58,-3576.0,19.798,13.2105,-1.755188,-0.96,21.205413,-8.626258,-13.2,20.3,3.16
10,2010,8931.519313,2.555336,26.21,64.3,17.0,3.57,5.27,35.74,806.0,19.639,14.0410,-2.029481,3.01,20.110710,-7.016844,-28.9,20.5,3.10
11,2011,10259.331961,8.752239,28.35,66.9,-15.0,3.73,5.23,32.59,-532.0,19.943,14.3203,-2.706873,4.36,18.467399,-5.541350,-16.2,19.2,3.03
12,2012,11120.403233,5.743976,28.20,68.5,16.0,4.23,5.17,26.68,-81.0,21.437,14.4911,0.613211,1.59,17.543879,-2.893765,-9.7,18.6,2.99
13,2013,11776.792571,5.026825,27.62,69.9,8.0,4.32,5.02,21.90,-892.0,22.689,13.3644,1.457239,1.66,16.735544,-2.417272,-9.0,20.6,2.96
14,2014,12395.979849,4.620353,27.73,71.8,6.0,4.19,4.94,19.26,-452.0,23.592,13.2117,1.930309,0.74,16.691874,-1.911556,-7.2,19.1,2.94
15,2015,12861.323296,3.756324,26.96,73.3,-22.0,4.12,4.96,16.30,260.0,25.748,13.6606,-0.986813,2.50,17.168991,-1.093729,-10.7,22.2,2.91
16,2016,13491.939909,3.877441,26.00,75.2,-6.0,4.40,4.90,14.45,25.0,25.612,13.5202,0.728400,2.74,17.029185,-0.460837,-12.7,21.9,2.88


In [55]:
fill_with_mean(df_lt, ["EMP","PAG", "QOR", "QOA", "GGAG"])

In [56]:
lt_context = needed_context(lt_context, context)

In [57]:
missing_values(lt_context)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
SSB - 2 - 12.5 %
PT - 2 - 12.5 %


In [58]:
# fill_with_mode(lt_context, ["DBD"])

In [59]:
fill_with_mean(lt_context, ["SSB", "PT"])

In [60]:
lt_payments = eu_payments_for_country(eu_payments, "LT")

In [61]:
lt = merge_data(df_lt, lt_context, lt_payments)

In [62]:
lt = fix_eu_payments(lt)

In [63]:
lt

,GDPC,GDPCG,VABI,EMP,PAG,QOA,QOR,YUA,GGAG,REST,...,LBGD,PC,ARP,ROL,GE,COC,RQ,SSB,PT,EUPC
0,8985.274630,12.405345,29.65,73.0,-3.000000,4.400000,4.970000,8.39,1013.000000,16.482,...,-0.493558,-11.6,19.1,0.69,0.66,0.10,1.01,81.367724,79.207745,45.521991
1,10219.116755,3.660125,29.20,73.0,25.000000,4.610000,5.210000,13.35,-563.000000,17.824,...,-2.844590,-9.0,20.9,0.68,0.56,0.11,1.04,81.503460,79.492010,70.724103
2,8528.034573,-13.887757,25.02,67.0,4.000000,4.220000,5.280000,29.58,-3576.000000,19.798,...,-8.626258,-13.2,20.3,0.72,0.69,0.23,0.95,82.186109,79.791757,337.935130
3,8931.519313,2.555336,26.21,64.3,17.000000,3.570000,5.270000,35.74,806.000000,19.639,...,-7.016844,-28.9,20.5,0.78,0.74,0.37,0.96,82.520151,80.789148,281.231582
4,10259.331961,8.752239,28.35,66.9,-15.000000,3.730000,5.230000,32.59,-532.000000,19.943,...,-5.541350,-16.2,19.2,0.77,0.70,0.32,0.93,84.636537,80.127684,309.501994
5,11120.403233,5.743976,28.20,68.5,16.000000,4.230000,5.170000,26.68,-81.000000,21.437,...,-2.893765,-9.7,18.6,0.84,0.83,0.39,1.12,84.670001,81.101556,339.126224
6,11776.792571,5.026825,27.62,69.9,8.000000,4.320000,5.020000,21.90,-892.000000,22.689,...,-2.417272,-9.0,20.6,0.83,0.83,0.42,1.15,84.317188,81.028515,362.332974
7,12395.979849,4.620353,27.73,71.8,6.000000,4.190000,4.940000,19.26,-452.000000,23.592,...,-1.911556,-7.2,19.1,0.94,0.98,0.55,1.19,90.098078,81.075004,344.608645
8,12861.323296,3.756324,26.96,73.3,-22.000000,4.120000,4.960000,16.30,260.000000,25.748,...,-1.093729,-10.7,22.2,0.98,1.15,0.58,1.26,92.436369,81.549466,29.635159
9,13491.939909,3.877441,26.00,75.2,-6.000000,4.400000,4.900000,14.45,25.000000,25.612,...,-0.460837,-12.7,21.9,1.00,1.03,0.68,1.12,92.931332,85.480409,115.573864


In [64]:
lt.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_bn.csv')

In [65]:
lt_scaled = standardize_data(lt)

In [66]:
lt_scaled.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_tcdf.csv')

In [126]:
lt_scaled.columns

Index(['GDPC', 'GDPCG', 'VABI', 'EMP', 'PAG', 'QOA', 'QOR', 'YUA', 'GGAG',
       'REST', 'CO2', 'EBSG', 'FDI', 'GGFC', 'LBGD', 'PC', 'ARP', 'ROL', 'GE',
       'COC', 'RQ', 'SSB', 'PT', 'EUPC'],
      dtype='object')

In [ ]:
column_index = lt_scaled.columns.get_loc('EUPC')
print(column_index)

In [ ]:
9,1,0
23,4,3
8,4,0
5,4,0
20,4,3
13,5,0
11,7,0
12,8,0
10,8,0
22,11,0
13,14,0
1,15,1
16,16,4
23,16,1
12,23,3
22,23,0
11,23,3

In [67]:
df_lv = y_data(df_lv)

In [68]:
missing_values(df_lv)

TIME_PERIOD - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
VABI - 0 - 0.0 %
EMP - 2 - 12.5 %
PAG - 1 - 6.2 %
QOA - 3 - 18.8 %
QOR - 3 - 18.8 %
YUA - 0 - 0.0 %
GGAG - 2 - 12.5 %
REST - 0 - 0.0 %
CO2 - 0 - 0.0 %
EBSG - 0 - 0.0 %
FDI - 0 - 0.0 %
GGFC - 0 - 0.0 %
LBGD - 0 - 0.0 %
PC - 0 - 0.0 %
ARP - 0 - 0.0 %
PS - 0 - 0.0 %


In [69]:
fill_with_mean(df_lv, ["EMP", "PAG", "GGAG", "QOR", "QOA"])

In [70]:
lv_context = needed_context(lv_context, context)

In [71]:
missing_values(lv_context)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
SSB - 2 - 12.5 %
PT - 2 - 12.5 %


In [72]:
lv_context

,TIME_PERIOD,ROL,GE,COC,RQ,SSB,PT
7,2007,0.72,0.43,0.34,0.95,88.154100,81.115239
8,2008,0.78,0.52,0.23,0.97,88.467773,83.228626
9,2009,0.79,0.62,0.20,0.92,89.002428,83.043921
10,2010,0.76,0.72,0.20,0.93,89.196552,82.613206
11,2011,0.74,0.69,0.28,0.95,89.163024,81.896887
12,2012,0.78,0.83,0.24,1.01,91.495552,82.315191
13,2013,0.76,0.88,0.32,1.04,91.532399,84.229882
14,2014,0.87,0.95,0.41,1.17,92.332633,84.865341
15,2015,0.77,1.05,0.43,1.07,92.038243,86.462455
16,2016,0.93,0.97,0.40,1.07,94.107515,89.093857


In [73]:
fill_with_mean(lv_context, ["SSB", "PT"])

In [74]:
lv_payments = eu_payments_for_country(eu_payments, "LV")

In [75]:
lv = merge_data(df_lv, lv_context, lv_payments)

In [76]:
lv = fix_eu_payments(lv)

In [77]:
lv.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn.csv')

In [78]:
lv_scaled = standardize_data(lv)

In [79]:
lv_scaled.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_tcdf.csv')

In [128]:
lv_scaled.columns

Index(['GDPC', 'GDPCG', 'VABI', 'EMP', 'PAG', 'QOA', 'QOR', 'YUA', 'GGAG',
       'REST', 'CO2', 'EBSG', 'FDI', 'GGFC', 'LBGD', 'PC', 'ARP', 'ROL', 'GE',
       'COC', 'RQ', 'SSB', 'PT', 'EUPC'],
      dtype='object')

In [ ]:
column_index = lv_scaled.columns.get_loc('EUPC')
print(column_index)

In [ ]:
10,1,0
8,4,3
17,4,0
23,5,0
23,6,2
12,6,3
10,8,0
21,12,0
1,12,1
2,12,3
1,14,1
13,14,0
18,16,1
20,23,0

In [80]:
df_ee = y_data(df_ee)

In [81]:
missing_values(df_ee)

TIME_PERIOD - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
VABI - 0 - 0.0 %
EMP - 2 - 12.5 %
PAG - 1 - 6.2 %
QOA - 3 - 18.8 %
QOR - 3 - 18.8 %
YUA - 0 - 0.0 %
GGAG - 2 - 12.5 %
REST - 0 - 0.0 %
CO2 - 0 - 0.0 %
EBSG - 0 - 0.0 %
FDI - 0 - 0.0 %
GGFC - 0 - 0.0 %
LBGD - 0 - 0.0 %
PC - 0 - 0.0 %
ARP - 0 - 0.0 %
PS - 0 - 0.0 %


In [82]:
fill_with_mode(df_ee, ["QOR", "QOA"])

In [83]:
fill_with_mean(df_ee, ["EMP", "PAG", "GGAG"])

In [84]:
ee_context = needed_context(ee_context, context)

In [85]:
missing_values(ee_context)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
SSB - 2 - 12.5 %
PT - 2 - 12.5 %


In [86]:
# fill_with_mode(ee_context, ["DBD"])

In [87]:
fill_with_mean(ee_context, ["SSB", "PT"])

In [88]:
ee_payments = eu_payments_for_country(eu_payments, "EE")

In [89]:
ee = merge_data(df_ee, ee_context, ee_payments)

In [90]:
ee = fix_eu_payments(ee)

In [91]:
ee.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/ee_bn.csv')

In [92]:
ee_scaled = standardize_data(ee)

In [93]:
ee_scaled.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/ee_tcdf.csv')

In [94]:
ee_scaled

,GDPC,GDPCG,VABI,EMP,PAG,QOA,QOR,YUA,GGAG,REST,...,LBGD,PC,ARP,ROL,GE,COC,RQ,SSB,PT,EUPC
0,-0.953544,1.148225,2.236931,0.000000,5.616224e-01,1.171421,-2.386756,-1.200121,1.681745,-1.900719,...,2.110672,-0.637588,-0.375920,-1.198330,-0.781556,-1.241310,-1.632933,-3.211033e+00,-0.129121,-1.632853
1,-0.911800,-1.223509,1.701951,0.000000,1.204701e+00,1.456580,-1.607407,-0.879507,-0.625070,-1.596798,...,-0.822032,-0.444829,-0.323160,-0.786002,0.452480,-1.333046,-1.027342,-3.918685e-01,-0.035113,-1.438183
2,-1.305592,-2.983038,-0.445502,-1.433676,9.474698e-01,0.860338,-0.828058,1.675435,-1.105577,-0.833725,...,-0.942914,-0.415174,-0.217638,-1.198330,-0.863825,-1.103705,-1.027342,-2.962680e-01,0.038600,0.345246
3,-1.202445,0.161798,0.119617,-2.182804,5.616224e-01,0.030784,0.270115,2.594084,1.987732,-0.549072,...,-0.022831,-0.548622,-2.275308,-0.889084,-0.205673,-1.149573,-1.027342,-2.853289e-01,0.006063,0.607980
4,-0.872622,1.124639,1.114227,-1.304516,-1.367615e+00,-0.383993,0.128415,0.841507,0.234075,-0.378208,...,0.618564,-0.637588,-1.378375,-0.579837,-0.287942,-0.874364,-1.027342,-3.399175e-01,-0.053625,-0.815741
5,-0.645259,0.411217,0.571712,-0.762044,-2.653773e+00,-0.202528,-0.828058,0.587342,-0.319468,-0.365302,...,0.251857,-0.711726,-1.378375,-0.889084,-1.275170,-0.690891,-0.854316,-3.137592e-01,-1.434389,1.277491
6,-0.459909,0.058301,0.609387,-0.503724,3.686987e-01,-1.032082,-0.898908,0.228520,0.964445,-0.407109,...,0.475846,-0.637588,-0.798007,-0.476755,-1.110632,-0.278076,-0.594777,-2.637139e-01,-2.901003,1.375962
7,-0.255893,0.328758,0.179896,-0.271236,1.269009e+00,-1.965329,-0.225834,-0.389450,-0.226058,-0.266419,...,0.872031,-0.430001,0.890338,1.378724,-0.781556,0.272343,1.395023,-2.320716e-01,-0.317775,-0.111659
8,-0.153027,-0.005491,-0.566061,0.167908,-8.531516e-01,-2.094947,0.270115,-0.495768,-0.985258,0.252900,...,0.389182,-0.029655,0.784816,0.657149,-0.534749,0.088870,1.308510,4.039082e-01,-0.272377,-1.471272
9,0.037684,0.231363,-0.483177,0.245404,-1.714877e-02,-0.098834,0.872339,-0.530653,0.786850,0.297434,...,0.218928,-0.177932,0.837577,-0.476755,-0.452480,-0.048735,1.568049,9.474251e-01,0.709937,-1.035978


In [130]:
ee_scaled.columns

Index(['GDPC', 'GDPCG', 'VABI', 'EMP', 'PAG', 'QOA', 'QOR', 'YUA', 'GGAG',
       'REST', 'CO2', 'EBSG', 'FDI', 'GGFC', 'LBGD', 'PC', 'ARP', 'ROL', 'GE',
       'COC', 'RQ', 'SSB', 'PT', 'EUPC'],
      dtype='object')

In [131]:
column_index = ee_scaled.columns.get_loc('EUPC')
print(column_index)

23


In [ ]:
3,2,0
17,4,1
17,5,3
14,7,0
10,8,1
14,9,1
23,10,2
6,11,0
9,11,0
14,12,2
22,12,0
14,13,0
10,14,3
13,14,0
12,15,0
14,18,0
17,20,2
6,21,0
21,23,0

In [95]:
df_fi = y_data(df_fi)

In [96]:
missing_values(df_fi)

TIME_PERIOD - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
VABI - 0 - 0.0 %
EMP - 2 - 12.5 %
PAG - 1 - 6.2 %
QOA - 3 - 18.8 %
QOR - 3 - 18.8 %
YUA - 0 - 0.0 %
GGAG - 2 - 12.5 %
REST - 0 - 0.0 %
CO2 - 0 - 0.0 %
EBSG - 0 - 0.0 %
FDI - 0 - 0.0 %
GGFC - 0 - 0.0 %
LBGD - 0 - 0.0 %
PC - 0 - 0.0 %
ARP - 0 - 0.0 %
PS - 0 - 0.0 %


In [97]:
fill_with_mode(df_fi, ["QOR"])

In [98]:
fill_with_mean(df_fi, ["EMP", "PAG", "GGAG", "QOA"])

In [99]:
fi_context = needed_context(fi_context, context)

In [100]:
missing_values(fi_context)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
SSB - 2 - 12.5 %
PT - 2 - 12.5 %


In [101]:
fill_with_mean(fi_context, ["SSB", "PT"])

In [102]:
fi_payments = eu_payments_for_country(eu_payments, "FI")

In [103]:
fi = merge_data(df_fi, fi_context, fi_payments)

In [104]:
fi = fix_eu_payments(fi)

In [105]:
fi.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/fi_bn.csv')

In [106]:
fi_scaled = standardize_data(fi)

In [107]:
fi_scaled.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/fi_tcdf.csv')

In [132]:
fi_scaled.columns

Index(['GDPC', 'GDPCG', 'VABI', 'EMP', 'PAG', 'QOA', 'QOR', 'YUA', 'GGAG',
       'REST', 'CO2', 'EBSG', 'FDI', 'GGFC', 'LBGD', 'PC', 'ARP', 'ROL', 'GE',
       'COC', 'RQ', 'SSB', 'PT', 'EUPC'],
      dtype='object')

In [ ]:
column_index = fi_scaled.columns.get_loc('EUPC')
print(column_index)

In [ ]:
20,2,0
5,4,2
17,5,1
8,8,1
12,8,0
23,8,1
8,12,0
1,12,1
12,12,1
13,14,0
12,19,1
11,20,2
14,20,0
14,21,1
19,21,0
19,22,0
17,23,3

In [108]:
df_se = y_data(df_se)

In [109]:
missing_values(df_se)

TIME_PERIOD - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
VABI - 0 - 0.0 %
EMP - 0 - 0.0 %
PAG - 1 - 6.2 %
QOA - 3 - 18.8 %
QOR - 3 - 18.8 %
YUA - 0 - 0.0 %
GGAG - 2 - 12.5 %
REST - 0 - 0.0 %
CO2 - 0 - 0.0 %
EBSG - 0 - 0.0 %
FDI - 0 - 0.0 %
GGFC - 0 - 0.0 %
LBGD - 0 - 0.0 %
PC - 0 - 0.0 %
ARP - 0 - 0.0 %
PS - 0 - 0.0 %


In [110]:
fill_with_mode(df_se, ["QOR"])

In [111]:
fill_with_mean(df_se, ["GGAG", "QOA", "PAG"])

In [112]:
se_context = needed_context(se_context, context)

In [113]:
missing_values(se_context)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
SSB - 2 - 12.5 %
PT - 2 - 12.5 %


In [114]:
fill_with_mean(se_context, ["SSB", "PT"])

In [115]:
se_payments = eu_payments_for_country(eu_payments, "SE")

In [116]:
se = merge_data(df_se, se_context, se_payments)

In [117]:
se = fix_eu_payments(se)

In [118]:
se.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn.csv')

In [119]:
se_scaled = standardize_data(se)

In [120]:
se_scaled.to_csv('/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_tcdf.csv')

In [133]:
se_scaled.columns

Index(['GDPC', 'GDPCG', 'VABI', 'EMP', 'PAG', 'QOA', 'QOR', 'YUA', 'GGAG',
       'REST', 'CO2', 'EBSG', 'FDI', 'GGFC', 'LBGD', 'PC', 'ARP', 'ROL', 'GE',
       'COC', 'RQ', 'SSB', 'PT', 'EUPC'],
      dtype='object')

In [134]:
column_index = se_scaled.columns.get_loc('EUPC')
print(column_index)

23


In [ ]:
14,1,1
8,4,0
20,5,3
17,6,2
15,7,2
14,7,0
16,8,1
8,8,1
0,9,0
18,11,1
23,12,0
14,13,3
16,13,0
17,15,0
23,15,1
9,16,0
0,16,0
0,17,0
16,20,0 
0,20,1